In [1]:
import os
from datetime import datetime, date
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DecimalType, DateType, TimestampType
from clickhouse_driver import Client


In [2]:
# Configuration
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
CLICKHOUSE_HOST = "clickhouse"
CLICKHOUSE_PORT = 9000
CLICKHOUSE_USER = "default"
CLICKHOUSE_PASSWORD = ""
CLICKHOUSE_DB = "bronze"


In [3]:
# Spark Session
spark = (SparkSession.builder
    .appName("OracleToClickHouseIngestion")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate())


In [ ]:
# Schema Definition
schema_oracle = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("customer_id", IntegerType(), False),
    StructField("amount", DecimalType(18, 2), True),
    StructField("transaction_date", DateType(), False),
    StructField("created_at", TimestampType(), False),
    StructField("status", StringType(), True)
])


In [ ]:
# Mock Data Generation (Simulating Oracle Extraction)
# In production, this would be: spark.read.format("jdbc")...
data_volume = 100000
df_oracle = spark.range(0, data_volume).select(
    F.expr("uuid()").alias("transaction_id"),
    F.expr("cast(rand() * 10000 as int)").alias("customer_id"),
    F.expr("cast(rand() * 5000 as decimal(18,2))").alias("amount"),
    F.current_date().alias("transaction_date"),
    F.current_timestamp().alias("created_at"),
    F.expr("case when rand() > 0.9 then 'CANCELLED' else 'COMPLETED' end").alias("status")
)


In [ ]:
# Staging (Simulating Write to Shared Storage/Object Store)
staging_path = "/app/temp/staging_transactions"
df_oracle.write.mode("overwrite").parquet(staging_path)


In [ ]:
# ClickHouse Connection
client = Client(host=CLICKHOUSE_HOST, port=CLICKHOUSE_PORT, user=CLICKHOUSE_USER, password=CLICKHOUSE_PASSWORD)


In [ ]:
# DDL Execution (Bronze Layer)
client.execute(f"CREATE DATABASE IF NOT EXISTS {CLICKHOUSE_DB}")

ddl_bronze = f"""
CREATE TABLE IF NOT EXISTS {CLICKHOUSE_DB}.transactions_local
(
    transaction_id String,
    customer_id UInt32,
    amount Decimal(18,2),
    transaction_date Date,
    created_at DateTime,
    status String,
    ingestion_date Date DEFAULT today()
)
ENGINE = MergeTree()
PARTITION BY toYYYYMM(transaction_date)
ORDER BY (transaction_date, customer_id, transaction_id)
"""
client.execute(ddl_bronze)


In [ ]:
# Ingestion from Staging to ClickHouse
# Reading back from Parquet to simulate the decoupling
df_staging = spark.read.parquet(staging_path)

# Convert to list of tuples for ClickHouse driver
# For massive datasets, we would use clickhouse-client via subprocess or specialized format writer
data_to_insert = df_staging.collect()

insert_query = f"INSERT INTO {CLICKHOUSE_DB}.transactions_local (transaction_id, customer_id, amount, transaction_date, created_at, status) VALUES"
client.execute(insert_query, data_to_insert)


In [ ]:
# Validation
result = client.execute(f"SELECT count(), sum(amount) FROM {CLICKHOUSE_DB}.transactions_local")
print(f"Rows inserted: {result[0][0]}")
print(f"Total Amount: {result[0][1]}")

packages = [
    "com.clickhouse.spark:clickhouse-spark-runtime-3.4_2.12:0.8.0",
    "com.clickhouse:clickhouse-client:0.7.0",
    "com.clickhouse:clickhouse-http-client:0.7.0",
    "com.oracle.database.jdbc:ojdbc8:23.2.0.0",
    "org.apache.httpcomponents.client5:httpclient5:5.2.1"
]

spark = (
    SparkSession.builder
        .appName("OracleToClickHouse")
        .master("spark://spark-master:7077")
        .config("spark.jars.packages", ",".join(packages))
        .config("spark.sql.shuffle.partitions", "4")
        .getOrCreate()
)

In [ ]:
oracle_df = (
    spark.read.format("jdbc")
    .option("driver", "oracle.jdbc.OracleDriver")
    .option("url", "jdbc:oracle:thin:@//ORACLE_HOST:1521/SERVICE_NAME")
    .option("user", "ORACLE_USER")
    .option("password", "ORACLE_PASS")
    .option("dbtable", "SCHEMA.TABELA_FONTE")
    .load()
)

oracle_df.show(5)